# D: revision safety / 修改行为分析

本模板没有填入实验结果。使用 E 提供的独立正确性标签；不能把 D 自己的核验结果当作正确性真值。
输出是 D 的描述性机制分析，正式评分、置信区间及抽样规则由 E 负责。

将 E 的独立标注导出为 `research/independent_revision_labels.json`，内容为 JSON 数组。
每行包含：`example_id`、`method_id`（M3/M4/M5）、`status`（ok/error/skipped）、
`draft_correct` 和 `final_correct`（布尔值或 null）。可选加入 `draft_answer`、`final_answer`、
`failure_category`、`notes` 供案例分析。这里的标签表是 D 的分析输入，不改动共享 ExperimentRecord。

`error` / `skipped` 和缺少独立标签的记录单列，不冒充正确或错误。
如果只分析抽样案例，结果只描述该样本；报告样本量、抽样方式和标签来源。


In [ ]:
import json
from collections import Counter
from pathlib import Path

project_root = Path.cwd()
if project_root.name == 'research':
    project_root = project_root.parent
labels_path = project_root / 'research' / 'independent_revision_labels.json'
cases = json.loads(labels_path.read_text(encoding='utf-8')) if labels_path.exists() else []
if not isinstance(cases, list):
    raise ValueError('The independent label file must contain a JSON array.')
print(f'Loaded {len(cases)} records.' if cases else 'No independent labels yet; no research results to report.')


In [ ]:
def summarize(rows):
    counts = {method: Counter() for method in ('M3', 'M4', 'M5')}
    seen = set()
    for row in rows:
        method = row['method_id']
        if method not in counts:
            raise ValueError(f'Unexpected method: {method}')
        key = (row['example_id'], method)
        if key in seen:
            raise ValueError(f'Duplicate example/method: {key}')
        seen.add(key)
        status = row['status']
        if status not in ('ok', 'error', 'skipped'):
            raise ValueError(f'Unexpected status: {status}')
        counts[method]['total'] += 1
        counts[method][status] += 1
        if status != 'ok':
            continue
        before, after = row.get('draft_correct'), row.get('final_correct')
        if before is not None and type(before) is not bool:
            raise ValueError('draft_correct must be a bool or null.')
        if after is not None and type(after) is not bool:
            raise ValueError('final_correct must be a bool or null.')
        if before is None or after is None:
            counts[method]['unlabeled'] += 1
            continue
        counts[method]['labeled'] += 1
        transition = {
            (False, True): 'wrong_to_correct',
            (True, False): 'correct_to_wrong',
            (False, False): 'wrong_to_wrong',
            (True, True): 'correct_to_correct',
        }[(before, after)]
        counts[method][transition] += 1
    return counts

summary = summarize(cases)
if cases:
    columns = ('total', 'ok', 'error', 'skipped', 'unlabeled', 'labeled',
               'wrong_to_correct', 'correct_to_wrong', 'wrong_to_wrong', 'correct_to_correct')
    print('method\t' + '\t'.join(columns))
    for method, counts in summary.items():
        print(method + '\t' + '\t'.join(str(counts[column]) for column in columns))


## 案例解释

从 E 的正式记录中选择少量错→对与对→错案例，保留原稿、证据、反馈和最终答案。
可记录 `failure_category`：证据不足、核验误判、改写忽略反馈、过度修改、其他。
这些是人工机制解释，不替代 E 的独立正确性标签。

没有实际案例时保留本模板为空。独立标注的具体抽样与分工由 E 安排。


In [ ]:
for row in cases:
    if row.get('notes'):
        print({key: row.get(key) for key in (
            'example_id', 'method_id', 'draft_answer', 'final_answer',
            'draft_correct', 'final_correct', 'failure_category', 'notes')})
